# 03 Functional And Jacobian Analysis

This notebook asks whether the realized SPD function contracts relative to the target. In the new reframing,
these plots diagnose **functional contraction**, which is stronger evidence than raw weight ratios alone.


In [1]:
from pathlib import Path
import sys

if (Path.cwd() / 'koko_notebooks').exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = Path.cwd().resolve().parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from koko_notebooks.shrinkage_analysis.shrinkage_analysis_common import (
    OUTPUTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
    batched_expected_masked_weight,
    batched_expected_train_mask_weight,
    collect_ci_outputs,
    component_strengths,
    discover_exp07_analysis_jsons,
    ensure_outputs_dir,
    exhaustive_binary_probe_batch,
    latest_result_per_run,
    layer_weight_metric_row,
    load_component_model_for_checkpoint,
    save_dataframe,
    save_json,
    sampled_probe_batch,
    select_consistent_replicate,
    singleton_probe_batch,
)
from koko_notebooks.shrinkage_analysis.publication_plots import (
    ARCH_COLORS,
    LAYER_COLORS,
    architecture_comparison_plot,
    heatmap,
    histogram_triptych,
    line_plot_by_group,
    line_plot_by_layer,
    multi_metric_panel_by_group,
    multi_metric_panel_by_layer,
    parse_vector_column,
    save_figure,
    setup_publication_style,
    singular_value_trajectory_plot,
)

ensure_outputs_dir()
setup_publication_style()
OUTPUTS_DIR


/root/spd_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1')

In [2]:
import numpy as np
import pandas as pd
import torch
from torch.nn.functional import mse_loss
from tqdm.auto import tqdm

from spd.models.components import make_mask_infos

CONSISTENT_REPLICATE = 1

manifest_df = latest_result_per_run(discover_exp07_analysis_jsons())
manifest_df = select_consistent_replicate(manifest_df, CONSISTENT_REPLICATE)
manifest_df = manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).reset_index(drop=True)

SELECTED_DEPTHS = [2, 3, 4, 5, 6]
SELECTED_ARCHITECTURES = ['tied', 'untied']
PROBE_TYPE = 'exhaustive'
INPUT_MAGNITUDE = 1.0
SAMPLED_BATCH_SIZE = 256
SAMPLED_SEED = 0
DEVICE = 'cpu'

selected_manifest_df = manifest_df[
    manifest_df['depth'].isin(SELECTED_DEPTHS) & manifest_df['architecture'].isin(SELECTED_ARCHITECTURES)
].copy()
selected_manifest_df[['run_name', 'depth', 'architecture', 'replicate']]


,run_name,depth,architecture,replicate
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1
1,exp_07_tms_5_2_2layer_untied_rep1,2,untied,1
2,exp_07_tms_5_2_3layer_tied_rep1,3,tied,1
3,exp_07_tms_5_2_3layer_untied_rep1,3,untied,1
4,exp_07_tms_5_2_4layer_tied_rep1,4,tied,1
5,exp_07_tms_5_2_4layer_untied_rep1,4,untied,1
6,exp_07_tms_5_2_5layer_tied_rep1,5,tied,1
7,exp_07_tms_5_2_5layer_untied_rep1,5,untied,1
8,exp_07_tms_5_2_6layer_tied_rep1,6,tied,1
9,exp_07_tms_5_2_6layer_untied_rep1,6,untied,1


In [3]:
def make_probe_batch(config, device: str):
    task_config = config.task_config
    if PROBE_TYPE == 'singleton':
        return singleton_probe_batch(n_features=5, input_magnitude=INPUT_MAGNITUDE, device=device)
    if PROBE_TYPE == 'exhaustive':
        return exhaustive_binary_probe_batch(n_features=5, device=device)
    return sampled_probe_batch(task_config=task_config, batch_size=SAMPLED_BATCH_SIZE, device=device, seed=SAMPLED_SEED)

def raw_spd_forward(component_model, batch):
    ones_mask_infos = make_mask_infos({layer_name: torch.ones(component_model.module_to_c[layer_name], device=batch.device) for layer_name in component_model.target_module_paths})
    return component_model(batch, mask_infos=ones_mask_infos)

def ci_masked_forward(component_model, batch, sampling: str):
    pre_weight_acts = component_model(batch, cache_type='input').cache
    ci_outputs = component_model.calc_causal_importances(pre_weight_acts=pre_weight_acts, sampling=sampling)
    mask_infos = make_mask_infos(ci_outputs.lower_leaky)
    return component_model(batch, mask_infos=mask_infos)

def jacobian_fro_norm(fn, x: torch.Tensor) -> float:
    jac = torch.autograd.functional.jacobian(fn, x)
    return float(jac.norm().item())

def jacobian_ratio_summary(component_model, target_model, sampling: str, probe_points: torch.Tensor) -> tuple[float, float]:
    raw_ratios = []
    ci_ratios = []
    for probe_x in probe_points:
        x = probe_x.clone().detach().requires_grad_(True)

        def target_single(inp):
            return target_model(inp.unsqueeze(0)).squeeze(0)

        def raw_single(inp):
            return raw_spd_forward(component_model, inp.unsqueeze(0)).squeeze(0)

        def ci_single(inp):
            return ci_masked_forward(component_model, inp.unsqueeze(0), sampling=sampling).squeeze(0)

        target_norm = max(jacobian_fro_norm(target_single, x), 1e-12)
        raw_ratios.append(jacobian_fro_norm(raw_single, x) / target_norm)
        ci_ratios.append(jacobian_fro_norm(ci_single, x) / target_norm)
    return float(np.mean(raw_ratios)), float(np.mean(ci_ratios))


In [4]:
functional_rows: list[dict[str, object]] = []
jacobian_probe_points = singleton_probe_batch(n_features=5, input_magnitude=INPUT_MAGNITUDE, device=DEVICE)

for _, manifest_row in selected_manifest_df.iterrows():
    spd_run_dir = Path(manifest_row['spd_run_dir'])
    for step in tqdm(manifest_row['checkpoint_steps'], desc=manifest_row['run_name']):
        component_model, target_model, config = load_component_model_for_checkpoint(spd_run_dir=spd_run_dir, step=int(step), device=DEVICE)
        probe_batch = make_probe_batch(config=config, device=DEVICE)
        with torch.no_grad():
            target_out = target_model(probe_batch)
            raw_out = raw_spd_forward(component_model, probe_batch)
            ci_out = ci_masked_forward(component_model, probe_batch, sampling=config.sampling)
        raw_jac_ratio, ci_jac_ratio = jacobian_ratio_summary(component_model=component_model, target_model=target_model, sampling=config.sampling, probe_points=jacobian_probe_points)
        functional_rows.append(
            {
                'run_name': manifest_row['run_name'],
                'depth': int(manifest_row['depth']),
                'architecture': manifest_row['architecture'],
                'replicate': int(manifest_row['replicate']),
                'checkpoint_step': int(step),
                'probe_type': PROBE_TYPE,
                'raw_output_norm_ratio_mean': float((raw_out.norm(dim=-1) / target_out.norm(dim=-1).clamp_min(1e-12)).mean().item()),
                'ci_masked_output_norm_ratio_mean': float((ci_out.norm(dim=-1) / target_out.norm(dim=-1).clamp_min(1e-12)).mean().item()),
                'raw_output_contraction_gap': float(1.0 - (raw_out.norm(dim=-1) / target_out.norm(dim=-1).clamp_min(1e-12)).mean().item()),
                'ci_output_contraction_gap': float(1.0 - (ci_out.norm(dim=-1) / target_out.norm(dim=-1).clamp_min(1e-12)).mean().item()),
                'raw_output_mse': float(mse_loss(raw_out, target_out).item()),
                'ci_masked_output_mse': float(mse_loss(ci_out, target_out).item()),
                'raw_jacobian_fro_ratio_mean': raw_jac_ratio,
                'ci_jacobian_fro_ratio_mean': ci_jac_ratio,
                'raw_jacobian_contraction_gap': float(1.0 - raw_jac_ratio),
                'ci_jacobian_contraction_gap': float(1.0 - ci_jac_ratio),
            }
        )

functional_df = pd.DataFrame(functional_rows)
functional_df.head()


exp_07_tms_5_2_2layer_tied_rep1:   0%|          | 0/8 [00:00<?, ?it/s]

exp_07_tms_5_2_2layer_tied_rep1:  12%|█▎        | 1/8 [00:00<00:06,  1.12it/s]

exp_07_tms_5_2_2layer_tied_rep1:  25%|██▌       | 2/8 [00:01<00:02,  2.14it/s]

exp_07_tms_5_2_2layer_tied_rep1:  38%|███▊      | 3/8 [00:01<00:01,  2.91it/s]

exp_07_tms_5_2_2layer_tied_rep1:  50%|█████     | 4/8 [00:01<00:01,  3.95it/s]

...

exp_07_tms_5_2_6layer_untied_rep1:  75%|███████▌  | 6/8 [00:02<00:00,  2.55it/s]

exp_07_tms_5_2_6layer_untied_rep1:  88%|████████▊ | 7/8 [00:02<00:00,  2.74it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:03<00:00,  2.27it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:03<00:00,  2.41it/s]

,run_name,depth,architecture,replicate,checkpoint_step,probe_type,raw_output_norm_ratio_mean,ci_masked_output_norm_ratio_mean,raw_output_contraction_gap,ci_output_contraction_gap,raw_output_mse,ci_masked_output_mse,raw_jacobian_fro_ratio_mean,ci_jacobian_fro_ratio_mean,raw_jacobian_contraction_gap,ci_jacobian_contraction_gap
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,exhaustive,0.943979,1.103698,0.056021,-0.103698,0.000030,0.015196,1.005131,2.979306,-0.005131,-1.979306
1,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,exhaustive,0.939873,1.021005,0.060127,-0.021005,0.000017,0.007432,1.001835,3.316093,-0.001835,-2.316093
2,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,15000,exhaustive,0.941094,0.931970,0.058906,0.068030,0.000064,0.005926,1.002870,3.299758,-0.002870,-2.299758
3,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,20000,exhaustive,0.940999,0.908877,0.059001,0.091123,0.000022,0.006070,1.002782,3.486120,-0.002782,-2.486120
4,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,25000,exhaustive,0.944369,0.871936,0.055631,0.128064,0.000033,0.003936,1.005440,3.630595,-0.005440,-2.630595


In [5]:
functional_csv = save_dataframe(functional_df, 'csv/functional_metrics.csv')
functional_csv


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1/csv/functional_metrics.csv')

In [6]:
functional_mean_df = (
    functional_df.groupby(['depth', 'architecture', 'checkpoint_step'], as_index=False)
    .agg(
        raw_output_norm_ratio_mean=('raw_output_norm_ratio_mean', 'mean'),
        ci_masked_output_norm_ratio_mean=('ci_masked_output_norm_ratio_mean', 'mean'),
        raw_output_contraction_gap=('raw_output_contraction_gap', 'mean'),
        ci_output_contraction_gap=('ci_output_contraction_gap', 'mean'),
        raw_output_mse=('raw_output_mse', 'mean'),
        ci_masked_output_mse=('ci_masked_output_mse', 'mean'),
        raw_jacobian_fro_ratio_mean=('raw_jacobian_fro_ratio_mean', 'mean'),
        ci_jacobian_fro_ratio_mean=('ci_jacobian_fro_ratio_mean', 'mean'),
        raw_jacobian_contraction_gap=('raw_jacobian_contraction_gap', 'mean'),
        ci_jacobian_contraction_gap=('ci_jacobian_contraction_gap', 'mean'),
    )
)

plot_manifest = {}
for depth, depth_df in functional_mean_df.groupby('depth', sort=True):
    plot_manifest[f'functional_ratio_depth{depth}'] = multi_metric_panel_by_group(
        df=depth_df,
        x_col='checkpoint_step',
        group_col='architecture',
        y_cols=['raw_output_norm_ratio_mean', 'ci_masked_output_norm_ratio_mean'],
        titles=['Raw output ratio', 'CI-masked output ratio'],
        subdir='functional',
        stem=f'functional_ratio_depth{depth}',
        hline_at_one=True,
    )
    plot_manifest[f'functional_contraction_depth{depth}'] = multi_metric_panel_by_group(
        df=depth_df,
        x_col='checkpoint_step',
        group_col='architecture',
        y_cols=['raw_jacobian_contraction_gap', 'ci_jacobian_contraction_gap'],
        titles=['Raw Jacobian contraction gap', 'CI-masked Jacobian contraction gap'],
        subdir='functional',
        stem=f'functional_contraction_depth{depth}',
        hline_at_one=False,
    )
    plot_manifest[f'functional_mse_depth{depth}'] = multi_metric_panel_by_group(
        df=depth_df,
        x_col='checkpoint_step',
        group_col='architecture',
        y_cols=['raw_output_mse', 'ci_masked_output_mse'],
        titles=['Raw output MSE', 'CI-masked output MSE'],
        subdir='functional',
        stem=f'functional_mse_depth{depth}',
        hline_at_one=False,
    )
len(plot_manifest)


15

In [7]:
final_functional_df = functional_df.sort_values('checkpoint_step').groupby('run_name', as_index=False).tail(1)
final_functional_mean_df = (
    final_functional_df.groupby(['architecture', 'depth'], as_index=False)
    .agg(
        raw_output_norm_ratio_mean=('raw_output_norm_ratio_mean', 'mean'),
        ci_masked_output_norm_ratio_mean=('ci_masked_output_norm_ratio_mean', 'mean'),
        raw_jacobian_contraction_gap=('raw_jacobian_contraction_gap', 'mean'),
        ci_jacobian_contraction_gap=('ci_jacobian_contraction_gap', 'mean'),
    )
)

plot_manifest = plot_manifest
for architecture in ['tied', 'untied']:
    arch_df = final_functional_mean_df[final_functional_mean_df['architecture'] == architecture].copy().sort_values('depth')
    matrix = arch_df[['raw_output_norm_ratio_mean', 'ci_masked_output_norm_ratio_mean', 'raw_jacobian_contraction_gap', 'ci_jacobian_contraction_gap']].to_numpy()
    plot_manifest[f'functional_heatmap_{architecture}'] = heatmap(
        matrix=matrix,
        row_labels=[str(depth) for depth in arch_df['depth']],
        col_labels=['raw output ratio', 'masked output ratio', 'raw jac gap', 'masked jac gap'],
        title=f'Final functional contraction summary | {architecture}',
        colorbar_label='Value',
        subdir='functional',
        stem=f'functional_heatmap_{architecture}',
        vmin=0.0,
        vmax=1.1,
        annotate=True,
    )

save_json(plot_manifest, 'plots/functional/manifest.json')
plot_manifest


{'functional_ratio_depth2': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_ratio_depth2.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_ratio_depth2.pdf'},
 'functional_contraction_depth2': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_contraction_depth2.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_contraction_depth2.pdf'},
 'functional_mse_depth2': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_mse_depth2.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_mse_depth2.pdf'},
 'functional_ratio_depth3': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functional/functional_ratio_depth3.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/functi